In [1]:
# %load_ext cudf.pandas
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt 
import time
import cupy as cp

In [2]:
data = pd.read_csv('/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/week1.csv')
scout = pd.read_csv('/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/pffScoutingData.csv')
plays = pd.read_csv('/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/plays.csv')
players = pd.read_csv('/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/players.csv')

# Let's merge these data into one set 
data = data.merge(scout, how='left')

In [4]:
data = pd.concat([data]*factor)

0.5896946424618363
1.196050362661481
1.7940755439922214
2.392100725322962
2.9901259066537023
3.5881510879844427
4.186176269315183
4.784201450645924
5.382226631976664
5.9802518133074045


In [4]:
### cell 0
# get ball snap indicies 
_idxs = (data
         .loc[data['event']=='ball_snap', 
              'frameId']
         .index
         .values)

# to get 500ms of movement after snap, get 5 rows (each row is 100ms of info)
x = [(_idxs+x).tolist() for x in range(0,6)]
idxs = [item for sublist in x for item in sublist] #the output x is a list of lists, so this is just to flatten the list

# filter for snap + 500ms of data only using our selected indicies
_df = data.loc[idxs]

                                                                                                                  
                                            Total time elapsed: 7.330 seconds                                     
                                          19 GPU function calls in 6.946 seconds                                  
                                          0 CPU function calls in 0.000 seconds                                   
                                                                                                                  
                                                          Stats                                                   
                                                                                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function                     ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ DataFrame                    │ 1          │ 0.000       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.__getitem__        │ 1          │ 0.002       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ OpsMixin.__eq__              │ 1          │ 0.028       │ 0.028       │ 0          │ 0.000       │ 0.000       │
│ _LocationIndexer.__getitem__ │ 2          │ 5.187       │ 2.593       │ 0          │ 0.000       │ 0.000       │
│ Series                       │ 1          │ 0.000       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ Index                        │ 1          │ 0.000       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ ndarray.__add__              │ 6          │ 0.719       │ 0.120       │ 0          │ 0.000       │ 0.000       │
│ ndarray.tolist               │ 6          │ 1.009       │ 0.168       │ 0          │ 0.000       │ 0.000       │
└──────────────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

In [ ]:
%%time
### cell 1
gid = 2021090900
pid = 97 
nid = 25511 
_ = _df.loc[(_df['gameId']==gid) & (_df['playId']==pid) & (_df['nflId']==nid)]